# LeetCode 1202 — Smallest String With Swaps bằng Python

Link đề: https://leetcode.com/problems/smallest-string-with-swaps/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, String, Depth-First Search, Breadth-First Search, Union-Find, Sorting

Skeleton LeetCode:

```python
class Solution:
    def smallestStringWithSwaps(self, s: str, pairs: list[list[int]]) -> str:
        ...
```

Ghi chú: mấu chốt là nhận ra **"được hoán đổi bao nhiêu lần tùy ý"** ⇒ trong mỗi **thành phần liên thông (connected component)** của đồ thị chỉ số, các ký tự có thể được **sắp xếp lại theo bất kỳ hoán vị nào**. Vậy chỉ cần: tìm các thành phần (DFS/BFS/**Union-Find**), rồi trong mỗi thành phần **sắp xếp** ký tự và đặt lại vào các vị trí đã sắp xếp.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Mô hình hóa đề thành **đồ thị**: đỉnh là chỉ số `0..n-1`, mỗi cặp `[a, b]` là một cạnh vô hướng.
2. Hiểu vì sao trong một thành phần liên thông, các phép hoán đổi tạo ra **mọi hoán vị** (giống việc sắp xếp nổi bọt chỉ bằng hoán đổi kề nhau).
3. Tìm thành phần liên thông bằng **DFS (lặp)**, **BFS** và **Union-Find (Disjoint Set Union)** với nén đường đi + hợp theo kích thước.
4. Ghép kết quả: sắp xếp chỉ số và ký tự của từng nhóm, ghép cặp nhỏ-với-nhỏ.
5. Tránh bẫy: chỉ hoán đổi tham lam từng cặp một lần, quên tính bắc cầu, đệ quy DFS quá sâu với `n = 10^5`.

## 2. Đề bài & ví dụ

Cho chuỗi `s` và mảng `pairs`, trong đó `pairs[i] = [a, b]` là hai chỉ số (từ 0) của chuỗi. Bạn được **hoán đổi** ký tự tại hai chỉ số của **bất kỳ** cặp nào, **bao nhiêu lần cũng được**.

Trả về chuỗi **nhỏ nhất theo thứ tự từ điển (lexicographically smallest)** có thể đạt được.

**Ví dụ 1:** `s = "dcab"`, `pairs = [[0,3],[1,2]]` → `"bacd"`.

**Ví dụ 2:** `s = "dcab"`, `pairs = [[0,3],[1,2],[0,2]]` → `"abcd"`.

**Ví dụ 3:** `s = "cba"`, `pairs = [[0,1],[1,2]]` → `"abc"`.

**Ràng buộc:** `1 <= s.length <= 10^5`; `0 <= pairs.length <= 10^5`; `0 <= pairs[i][0], pairs[i][1] < s.length`; `s` chỉ gồm chữ thường.

In [1]:
import random
import time
from collections import defaultdict, deque
from typing import List

examples = [
    ("dcab", [[0, 3], [1, 2]], "bacd"),
    ("dcab", [[0, 3], [1, 2], [0, 2]], "abcd"),
    ("cba", [[0, 1], [1, 2]], "abc"),
]
for s, pairs, want in examples:
    print(f"s={s!r}, pairs={pairs} -> {want!r}")

s='dcab', pairs=[[0, 3], [1, 2]] -> 'bacd'
s='dcab', pairs=[[0, 3], [1, 2], [0, 2]] -> 'abcd'
s='cba', pairs=[[0, 1], [1, 2]] -> 'abc'


## 3. Kiến thức nền

### 3.1 Thứ tự từ điển (lexicographic order)

So sánh hai chuỗi cùng độ dài: tìm vị trí **đầu tiên** khác nhau, chuỗi nào có ký tự nhỏ hơn ở đó thì nhỏ hơn. Python so sánh chuỗi đúng theo cách này. Hệ quả: muốn chuỗi nhỏ nhất, hãy đặt ký tự nhỏ nhất có thể vào vị trí **sớm nhất** có thể.

In [2]:
print("bacd" < "bcad", "abcd" < "abdc", min(["dcab", "bacd", "bcad"]))

True True bacd


### 3.2 Danh sách kề (adjacency list)

Biểu diễn đồ thị vô hướng: `adj[u]` là danh sách các đỉnh nối với `u`. Mỗi cạnh `[a, b]` thêm `b` vào `adj[a]` **và** `a` vào `adj[b]`.

In [3]:
n = 4
adj = [[] for _ in range(n)]
for a, b in [[0, 3], [1, 2], [0, 2]]:
    adj[a].append(b)
    adj[b].append(a)
print(adj)

[[3, 2], [2], [1, 0], [0]]


### 3.3 DFS bằng stack (không đệ quy)

DFS đệ quy với `n = 10^5` đỉnh có thể vượt giới hạn đệ quy của Python (mặc định ~1000). Dùng stack tường minh: đẩy đỉnh xuất phát, lặp `pop`, đánh dấu, đẩy hàng xóm chưa thăm.

In [4]:
def component_of(start, adj, seen):
    comp, stack = [], [start]
    seen[start] = True
    while stack:
        u = stack.pop()
        comp.append(u)
        for v in adj[u]:
            if not seen[v]:
                seen[v] = True
                stack.append(v)
    return comp


seen = [False] * 4
print(component_of(0, adj, seen), seen)

[0, 2, 1, 3] [True, True, True, True]


### 3.4 Union-Find (Disjoint Set Union — DSU)

Cấu trúc quản lý các nhóm rời nhau:
- `find(x)`: trả về "đại diện" (gốc) của nhóm chứa `x`. **Nén đường đi (path compression)**: trỏ thẳng các nút trên đường về gốc.
- `union(x, y)`: gộp hai nhóm. **Hợp theo kích thước (union by size)**: gắn cây nhỏ vào cây lớn.

Với cả hai tối ưu, mỗi thao tác gần như `O(1)` (chính xác là `O(α(n))`, hàm Ackermann ngược — không quá 4 trong thực tế).

In [5]:
class DSU:
    def __init__(self, n):
        self.parent = list(range(n))
        self.size = [1] * n

    def find(self, x):
        root = x
        while self.parent[root] != root:
            root = self.parent[root]
        while self.parent[x] != root:          # nén đường đi (lặp)
            self.parent[x], x = root, self.parent[x]
        return root

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        if self.size[ra] < self.size[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        self.size[ra] += self.size[rb]
        return True


d = DSU(5)
d.union(0, 3); d.union(1, 2); d.union(3, 2)
print([d.find(i) for i in range(5)])

[0, 0, 0, 0, 4]


### 3.5 Sắp xếp và ghép cặp

Trong một nhóm có chỉ số `[3, 0, 2]` và ký tự tương ứng `['b', 'd', 'a']`: sắp xếp chỉ số → `[0, 2, 3]`, sắp xếp ký tự → `['a', 'b', 'd']`, rồi gán lần lượt `res[0]='a'`, `res[2]='b'`, `res[3]='d'`.

In [6]:
s = "dcab"
idx = [3, 0, 2]
chars = sorted(s[i] for i in idx)
res = list(s)
for i, ch in zip(sorted(idx), chars):
    res[i] = ch
print(''.join(res))

acbd


## 4. Ý tưởng nền tảng / trực giác

**Ví dụ 3:** `s = "cba"`, cạnh `0–1`, `1–2`. Dù không có cạnh `0–2`, ta vẫn đưa `a` từ vị trí 2 về vị trí 0:
`cba` →(1,2)→ `cab` →(0,1)→ `acb` →(1,2)→ `abc`.

→ Các chỉ số **liên thông** (gián tiếp qua nhiều cạnh) cũng "đổi chỗ" được cho nhau. Thực ra, hoán đổi theo các cạnh của một cây khung đủ để tạo ra **mọi** hoán vị của nhóm.

**Ví dụ 1:** nhóm `{0, 3}` có ký tự `d, b` → sắp xếp → vị trí 0 nhận `b`, vị trí 3 nhận `d`; nhóm `{1, 2}` có `c, a` → vị trí 1 nhận `a`, vị trí 2 nhận `c`. Kết quả `"bacd"`.

In [7]:
s = list("cba")
for a, b in [(1, 2), (0, 1), (1, 2)]:
    s[a], s[b] = s[b], s[a]
    print(f"swap({a},{b}) -> {''.join(s)}")

swap(1,2) -> cab
swap(0,1) -> acb
swap(1,2) -> abc


## 5. Cách 1 — Brute Force / baseline: BFS trên không gian chuỗi

**Ý tưởng:** coi mỗi chuỗi là một trạng thái; từ một trạng thái, mỗi cặp tạo một trạng thái mới (hoán đổi). BFS/DFS thăm **mọi** chuỗi đạt được, trả về `min`. Tuyệt đối đúng (mô phỏng đúng định nghĩa) — dùng làm **oracle** cho `n ≤ 7`.

In [8]:
def brute(s: str, pairs: List[List[int]]) -> str:
    seen = {s}
    q = deque([s])
    while q:
        cur = q.popleft()
        for a, b in pairs:
            lst = list(cur)
            lst[a], lst[b] = lst[b], lst[a]
            nxt = ''.join(lst)
            if nxt not in seen:
                seen.add(nxt)
                q.append(nxt)
    return min(seen)


for s, pairs, want in examples:
    print(brute(s, pairs), want)
    assert brute(s, pairs) == want

bacd bacd
abcd abcd
abc abc


**Độ phức tạp:** số trạng thái có thể tới `n!` (mọi hoán vị) → `O(n! · |pairs| · n)`. Với `n = 10` đã là 3.6 triệu trạng thái; `n = 10^5` thì vô vọng.

**Nút thắt:** ta liệt kê từng hoán vị, trong khi chỉ cần biết **tập ký tự** của mỗi nhóm có thể đặt tùy ý.

In [9]:
for n in [5, 6, 7, 8]:
    s = ''.join(random.Random(n).choice('abcdefgh') for _ in range(n))
    pairs = [[i, i + 1] for i in range(n - 1)]
    t = time.perf_counter(); r = brute(s, pairs); print(n, r, f"{time.perf_counter() - t:.3f}s")

5 eeeee 0.000s
6 bbbbbb 0.000s
7 fffffff 0.000s
8 dddddddd 0.000s


## 6. Bước đột phá

**Insight 1 (bắc cầu):** nếu `i` đổi được với `j`, `j` đổi được với `k` thì ký tự ở `i` có thể đi tới `k` (qua `j`) mà vẫn trả các ký tự khác về chỗ cũ: `swap(i,j), swap(j,k), swap(i,j)` đổi chỗ `i` và `k`. → Quan hệ "có thể đổi chỗ" là **quan hệ tương đương**, các lớp chính là **thành phần liên thông**.

**Insight 2 (mọi hoán vị):** mọi hoán vị đều phân tích được thành tích các **chuyển vị (transposition)**; trong một thành phần, mọi chuyển vị `(i, k)` đều thực hiện được (insight 1). Vậy mỗi thành phần có thể **sắp xếp tùy ý**.

**Insight 3 (tham lam theo thứ tự từ điển):** các thành phần độc lập với nhau; trong một thành phần, đặt ký tự nhỏ nhất vào chỉ số nhỏ nhất, ký tự nhỏ thứ hai vào chỉ số nhỏ thứ hai... là tối ưu.

**Quy tắc:** tìm thành phần → sắp xếp ký tự trong thành phần → gán theo chỉ số tăng dần.

## 7. Các cách tiếp theo

### Cách 2 — DFS (stack) tìm thành phần + sắp xếp

In [10]:
def solve_dfs(s: str, pairs: List[List[int]]) -> str:
    n = len(s)
    adj = [[] for _ in range(n)]
    for a, b in pairs:
        adj[a].append(b)
        adj[b].append(a)
    seen = [False] * n
    res = list(s)
    for start in range(n):
        if seen[start]:
            continue
        comp, stack = [], [start]
        seen[start] = True
        while stack:
            u = stack.pop()
            comp.append(u)
            for v in adj[u]:
                if not seen[v]:
                    seen[v] = True
                    stack.append(v)
        comp.sort()
        chars = sorted(s[i] for i in comp)
        for i, ch in zip(comp, chars):
            res[i] = ch
    return ''.join(res)


for s, pairs, want in examples:
    assert solve_dfs(s, pairs) == want
print([solve_dfs(s, p) for s, p, _ in examples])

['bacd', 'abcd', 'abc']


**Độ phức tạp:** xây đồ thị và duyệt `O(n + m)` (với `m = |pairs|`), sắp xếp các nhóm tổng cộng `O(n log n)`. Bộ nhớ `O(n + m)`.

### Cách 3 — BFS tìm thành phần
Giống hệt Cách 2 nhưng dùng `deque` (hàng đợi). Thứ tự thăm khác nhưng tập đỉnh mỗi thành phần như nhau.

In [11]:
def solve_bfs(s: str, pairs: List[List[int]]) -> str:
    n = len(s)
    adj = defaultdict(list)
    for a, b in pairs:
        adj[a].append(b)
        adj[b].append(a)
    seen = [False] * n
    res = list(s)
    for start in range(n):
        if seen[start]:
            continue
        seen[start] = True
        q = deque([start])
        comp = []
        while q:
            u = q.popleft()
            comp.append(u)
            for v in adj[u]:
                if not seen[v]:
                    seen[v] = True
                    q.append(v)
        comp.sort()
        for i, ch in zip(comp, sorted(s[i] for i in comp)):
            res[i] = ch
    return ''.join(res)


for s, pairs, want in examples:
    assert solve_bfs(s, pairs) == want
print([solve_bfs(s, p) for s, p, _ in examples])

['bacd', 'abcd', 'abc']


### Cách 4 — Union-Find + gom nhóm + sắp xếp (lời giải chuẩn)

`union(a, b)` cho mọi cặp, sau đó gom `groups[find(i)].append(i)` — duyệt `i` tăng dần nên danh sách chỉ số của mỗi nhóm **tự tăng dần**, không cần sắp xếp chỉ số.

In [12]:
def solve_dsu(s: str, pairs: List[List[int]]) -> str:
    n = len(s)
    dsu = DSU(n)
    for a, b in pairs:
        dsu.union(a, b)
    groups = defaultdict(list)
    for i in range(n):
        groups[dsu.find(i)].append(i)
    res = [''] * n
    for idx in groups.values():
        for i, ch in zip(idx, sorted(s[i] for i in idx)):
            res[i] = ch
    return ''.join(res)


for s, pairs, want in examples:
    assert solve_dsu(s, pairs) == want
print([solve_dsu(s, p) for s, p, _ in examples])

['bacd', 'abcd', 'abc']


**Độ phức tạp:** `O((n + m)·α(n) + n log n)`, bộ nhớ `O(n)` (không cần lưu danh sách kề — ưu điểm so với DFS/BFS).

### Cách 5 — Union-Find + đếm ký tự (counting sort, 26 chữ cái)

Thay vì `sorted` ký tự, đếm tần suất 26 chữ trong mỗi nhóm, rồi phát lần lượt từ `'a'` tới `'z'` cho các chỉ số tăng dần. Sắp xếp ký tự trở thành `O(n + 26·số_nhóm)`.

In [13]:
def solve_dsu_counting(s: str, pairs: List[List[int]]) -> str:
    n = len(s)
    dsu = DSU(n)
    for a, b in pairs:
        dsu.union(a, b)
    groups = defaultdict(list)
    for i in range(n):
        groups[dsu.find(i)].append(i)
    res = [''] * n
    for idx in groups.values():
        cnt = [0] * 26
        for i in idx:
            cnt[ord(s[i]) - 97] += 1
        k = 0
        for c in range(26):
            for _ in range(cnt[c]):
                res[idx[k]] = chr(97 + c)
                k += 1
    return ''.join(res)


for s, pairs, want in examples:
    assert solve_dsu_counting(s, pairs) == want
print([solve_dsu_counting(s, p) for s, p, _ in examples])

['bacd', 'abcd', 'abc']


**Độ phức tạp:** `O((n + m)·α(n) + 26·số_nhóm + n)` ≈ tuyến tính. Bộ nhớ `O(n)`.

## 8. TRACE từng bước

Trace Cách 4 trên `s = "dcab"`, `pairs = [[0,3],[1,2],[0,2]]`:

| bước | union | parent | size |
|---|---|---|---|
| 0 | — | [0,1,2,3] | [1,1,1,1] |
| 1 | (0,3) | [0,1,2,0] | [2,1,1,1] |
| 2 | (1,2) | [0,1,1,0] | [2,2,1,1] |
| 3 | (0,2): find(0)=0, find(2)=1 → gắn 1 vào 0 | [0,0,1,0] | [4,2,1,1] |

Gom nhóm: mọi chỉ số về gốc `0` → `idx = [0,1,2,3]`, ký tự `d,c,a,b` → sắp xếp `a,b,c,d` → `"abcd"`.

In [14]:
def trace_dsu(s, pairs):
    n = len(s)
    dsu = DSU(n)
    print("ban đầu  parent =", dsu.parent, "size =", dsu.size)
    for a, b in pairs:
        ra, rb = dsu.find(a), dsu.find(b)
        merged = dsu.union(a, b)
        print(f"union({a},{b}) gốc {ra},{rb} {'gộp' if merged else 'đã chung nhóm'} -> parent = {dsu.parent}, size = {dsu.size}")
    groups = defaultdict(list)
    for i in range(n):
        groups[dsu.find(i)].append(i)
    res = list(s)
    for root, idx in groups.items():
        chars = sorted(s[i] for i in idx)
        print(f"nhóm gốc {root}: chỉ số {idx}, ký tự {[s[i] for i in idx]} -> {chars}")
        for i, ch in zip(idx, chars):
            res[i] = ch
    print("kết quả:", ''.join(res))
    return ''.join(res)


trace_dsu("dcab", [[0, 3], [1, 2], [0, 2]])
print()
trace_dsu("dcab", [[0, 3], [1, 2]])

ban đầu  parent = [0, 1, 2, 3] size = [1, 1, 1, 1]
union(0,3) gốc 0,3 gộp -> parent = [0, 1, 2, 0], size = [2, 1, 1, 1]
union(1,2) gốc 1,2 gộp -> parent = [0, 1, 1, 0], size = [2, 2, 1, 1]
union(0,2) gốc 0,1 gộp -> parent = [0, 0, 1, 0], size = [4, 2, 1, 1]
nhóm gốc 0: chỉ số [0, 1, 2, 3], ký tự ['d', 'c', 'a', 'b'] -> ['a', 'b', 'c', 'd']
kết quả: abcd

ban đầu  parent = [0, 1, 2, 3] size = [1, 1, 1, 1]
union(0,3) gốc 0,3 gộp -> parent = [0, 1, 2, 0], size = [2, 1, 1, 1]
union(1,2) gốc 1,2 gộp -> parent = [0, 1, 1, 0], size = [2, 2, 1, 1]
nhóm gốc 0: chỉ số [0, 3], ký tự ['d', 'b'] -> ['b', 'd']
nhóm gốc 1: chỉ số [1, 2], ký tự ['c', 'a'] -> ['a', 'c']
kết quả: bacd


'bacd'

## 9. Code nộp LeetCode

In [15]:
class Solution:
    def smallestStringWithSwaps(self, s: str, pairs: List[List[int]]) -> str:
        n = len(s)
        parent = list(range(n))
        size = [1] * n

        def find(x: int) -> int:
            root = x
            while parent[root] != root:
                root = parent[root]
            while parent[x] != root:
                parent[x], x = root, parent[x]
            return root

        for a, b in pairs:
            ra, rb = find(a), find(b)
            if ra != rb:
                if size[ra] < size[rb]:
                    ra, rb = rb, ra
                parent[rb] = ra
                size[ra] += size[rb]

        groups = defaultdict(list)
        for i in range(n):
            groups[find(i)].append(i)          # chỉ số tăng dần tự nhiên
        res = [''] * n
        for idx in groups.values():
            for i, ch in zip(idx, sorted(s[i] for i in idx)):
                res[i] = ch
        return ''.join(res)


sol = Solution()
for s, pairs, want in examples:
    print(sol.smallestStringWithSwaps(s, pairs))
    assert sol.smallestStringWithSwaps(s, pairs) == want

bacd
abcd
abc


## 10. Vì sao đúng

1. **Không thể ra khỏi nhóm:** mỗi phép hoán đổi di chuyển ký tự dọc theo một cạnh, nên **tập (multiset) ký tự** trong mỗi thành phần liên thông không đổi. Mọi chuỗi đạt được đều có, tại mỗi thành phần, một hoán vị của multiset ban đầu.
2. **Đạt được mọi hoán vị trong nhóm:** với `i, k` cùng thành phần, tồn tại đường `i = v0 – v1 – ... – vt = k`; dãy hoán đổi dọc đường đi rồi quay ngược lại thực hiện chuyển vị `(i, k)` mà giữ nguyên các vị trí khác. Mọi hoán vị là tích các chuyển vị ⇒ đạt được.
3. **Tối ưu từ điển:** với mỗi nhóm độc lập, xét vị trí nhỏ nhất của nhóm — đặt ký tự nhỏ nhất của nhóm vào đó là bắt buộc để tối ưu (nếu không, đổi lại sẽ cho chuỗi nhỏ hơn tại vị trí đầu tiên khác biệt). Quy nạp cho các vị trí tiếp theo ⇒ gán "chỉ số tăng dần ↔ ký tự tăng dần". Các nhóm không ảnh hưởng nhau nên ghép lại vẫn tối ưu.

## 11. Time / Space Complexity

Gọi `n = len(s)`, `m = len(pairs)`.

| Cách | Time | Space |
|---|---|---|
| 1. BFS chuỗi | `O(n!·m·n)` | `O(n!·n)` |
| 2. DFS + sort | `O(n + m + n log n)` | `O(n + m)` |
| 3. BFS + sort | `O(n + m + n log n)` | `O(n + m)` |
| 4. DSU + sort | `O((n + m)·α(n) + n log n)` | `O(n)` |
| 5. DSU + đếm | `O((n + m)·α(n) + n + 26·k)` | `O(n)` |

`k` = số nhóm. Worst case sort: một nhóm duy nhất cỡ `n`.

## 12. Bộ test

In [16]:
solvers = {
    "dfs": solve_dfs,
    "bfs": solve_bfs,
    "dsu": solve_dsu,
    "dsu_counting": solve_dsu_counting,
    "submit": Solution().smallestStringWithSwaps,
}
edge = [
    ("a", [], "a"),
    ("ba", [], "ba"),
    ("ba", [[0, 1]], "ab"),
    ("ba", [[1, 0], [0, 1], [0, 1]], "ab"),
    ("zyx", [[0, 0]], "zyx"),
    ("udyyek", [[3, 3], [3, 0], [5, 1], [3, 1], [3, 4], [3, 5]], "deykuy"),
    ("qdwyt", [[2, 3], [3, 2], [0, 1], [4, 0], [3, 2]], "dqwyt"),
]
for s, pairs, want in examples + edge:
    assert brute(s, pairs) == want, (s, pairs, brute(s, pairs))
    for name, f in solvers.items():
        assert f(s, pairs) == want, (name, s, pairs, f(s, pairs), want)
print("ví dụ + edge: OK")

ví dụ + edge: OK


In [17]:
rng = random.Random(1202)
for trial in range(600):
    n = rng.randint(1, 6)
    s = ''.join(rng.choice('abcd') for _ in range(n))
    pairs = [[rng.randrange(n), rng.randrange(n)] for _ in range(rng.randint(0, 5))]
    want = brute(s, pairs)
    for name, f in solvers.items():
        assert f(s, pairs) == want, (name, s, pairs)
print("600 test ngẫu nhiên khớp brute force")

600 test ngẫu nhiên khớp brute force


In [18]:
rng = random.Random(5)
n = m = 10**5
s = ''.join(rng.choice('abcdefghijklmnopqrstuvwxyz') for _ in range(n))
pairs = [[rng.randrange(n), rng.randrange(n)] for _ in range(m)]
chain = [[i, i + 1] for i in range(n - 1)]          # một thành phần cực dài (DFS đệ quy sẽ vỡ)
for name, f in solvers.items():
    t = time.perf_counter(); r1 = f(s, pairs); r2 = f(s, chain); dt = time.perf_counter() - t
    print(f"{name:13s} {dt:.3f}s")
assert solve_dsu(s, chain) == ''.join(sorted(s))
assert len({f(s, pairs) for f in solvers.values()}) == 1
print("test giới hạn OK")

dfs           0.246s


bfs           0.328s
dsu           0.196s


dsu_counting  0.236s
submit        0.190s


test giới hạn OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tham lam: duyệt từng cặp một lần, đổi nếu có lợi

In [19]:
def wrong_greedy_once(s, pairs):
    res = list(s)
    for a, b in pairs:
        i, j = min(a, b), max(a, b)
        if res[i] > res[j]:
            res[i], res[j] = res[j], res[i]
    return ''.join(res)


print(wrong_greedy_once("cba", [[0, 1], [1, 2]]), "đúng là", brute("cba", [[0, 1], [1, 2]]))

bac đúng là abc


**Vì sao sai:** đề cho đổi **bao nhiêu lần cũng được** và theo **bất kỳ thứ tự** nào; một lượt duyệt không đủ để đưa `a` từ cuối lên đầu. **Sửa:** dùng thành phần liên thông.

### Lỗi 2 — Lặp lại "đổi nếu có lợi" đến khi ổn định (vẫn sai!)

In [20]:
def wrong_greedy_fixpoint(s, pairs):
    res = list(s)
    changed = True
    while changed:
        changed = False
        for a, b in pairs:
            i, j = min(a, b), max(a, b)
            if res[i] > res[j]:
                res[i], res[j] = res[j], res[i]
                changed = True
    return ''.join(res)


rng = random.Random(13)
cex = None
for _ in range(5000):
    n = rng.randint(2, 6)
    s = ''.join(rng.choice('abc') for _ in range(n))
    pairs = [[rng.randrange(n), rng.randrange(n)] for _ in range(rng.randint(1, 4))]
    if wrong_greedy_fixpoint(s, pairs) != brute(s, pairs):
        cex = (s, pairs); break
print("phản ví dụ:", cex, "| sai:", wrong_greedy_fixpoint(*cex), "| đúng:", brute(*cex))

phản ví dụ: ('caa', [[1, 0], [1, 1], [0, 0], [2, 0]]) | sai: aca | đúng: aac


**Vì sao sai:** chỉ hoán đổi khi **có lợi ngay** có thể kẹt ở "cực tiểu địa phương": đôi khi phải đổi một cặp làm chuỗi **tạm thời lớn hơn** (đưa ký tự đi qua vị trí trung gian không có cạnh trực tiếp tới đích) rồi mới tốt hơn. Phản ví dụ tìm được ở trên: `s = "caa"`, cạnh `0–1`, `0–2`. Thuật toán đổi `(0,1)` được `"aca"` rồi kẹt: đổi `(0,2)` là `a↔a` vô ích, đổi `(0,1)` làm chuỗi lớn hơn. Nhưng `c` cần đi từ vị trí 1 sang 2 **qua** vị trí 0 (tạm thời thành `"caa"` rồi `"aac"`), nên đáp án là `"aac"`. **Sửa:** dùng thành phần liên thông.

### Lỗi 3 — DFS đệ quy với chuỗi dài

In [21]:
import sys


def dfs_recursive_components(n, pairs):
    adj = [[] for _ in range(n)]
    for a, b in pairs:
        adj[a].append(b); adj[b].append(a)
    seen = [False] * n

    def dfs(u, comp):
        seen[u] = True
        comp.append(u)
        for v in adj[u]:
            if not seen[v]:
                dfs(v, comp)

    comps = []
    for i in range(n):
        if not seen[i]:
            comp = []
            dfs(i, comp)
            comps.append(comp)
    return comps


try:
    dfs_recursive_components(5000, [[i, i + 1] for i in range(4999)])
    print("chạy được")
except RecursionError as e:
    print("RecursionError với chuỗi 5000 đỉnh; giới hạn đệ quy =", sys.getrecursionlimit())

RecursionError với chuỗi 5000 đỉnh; giới hạn đệ quy = 1000


**Vì sao sai:** đồ thị dạng đường thẳng dài `10^5` cần độ sâu đệ quy `10^5`, vượt giới hạn mặc định. **Sửa:** DFS bằng stack, BFS, hoặc Union-Find (không đệ quy). (Tăng `sys.setrecursionlimit` cũng có thể gây tràn stack C và crash.)

### Lỗi 4 — Sắp xếp ký tự nhưng gán theo thứ tự thăm DFS (không sắp chỉ số)

In [22]:
def wrong_unsorted_indices(s, pairs):
    n = len(s)
    adj = [[] for _ in range(n)]
    for a, b in pairs:
        adj[a].append(b); adj[b].append(a)
    seen = [False] * n
    res = list(s)
    for st in range(n):
        if seen[st]:
            continue
        comp, stack = [], [st]
        seen[st] = True
        while stack:
            u = stack.pop(); comp.append(u)
            for v in adj[u]:
                if not seen[v]:
                    seen[v] = True; stack.append(v)
        for i, ch in zip(comp, sorted(s[i] for i in comp)):   # SAI: comp chưa sắp xếp
            res[i] = ch
    return ''.join(res)


print(wrong_unsorted_indices("dcab", [[0, 3], [1, 2], [0, 2]]), "đúng là", brute("dcab", [[0, 3], [1, 2], [0, 2]]))

acbd đúng là abcd


**Vì sao sai:** thứ tự thăm DFS (`0, 2, 1, 3` chẳng hạn) không phải thứ tự chỉ số; ký tự nhỏ nhất phải về **chỉ số nhỏ nhất**. **Sửa:** `comp.sort()` trước khi ghép (DSU gom theo `i` tăng dần thì tự đúng).

## 14. Mẹo & nhận dạng pattern

- **"Được hoán đổi các cặp bao nhiêu lần tùy ý"** → thành phần liên thông = các nhóm sắp xếp tự do.
- **Quan hệ bắc cầu, gom nhóm** → Union-Find là lựa chọn gọn nhất (không cần danh sách kề, không đệ quy).
- **Nhỏ nhất theo từ điển** → tham lam vị trí sớm nhất nhận ký tự nhỏ nhất.
- **Bảng chữ cái nhỏ** → counting sort thay `sorted`.
- Trong Python, tránh đệ quy sâu với đồ thị `10^5` đỉnh.

## 15. Biến thể

1. **LeetCode 1722 — Minimize Hamming Distance After Swap Operations:** cùng khung DSU, so multiset nhóm với `target`.
2. **LeetCode 2948 — Make Lexicographically Smallest Array by Swapping Elements:** cạnh ẩn (hai số chênh lệch `≤ limit`) → sắp xếp rồi chia nhóm liên tiếp.
3. **LeetCode 839 / 990** — gom nhóm bằng DSU với quan hệ khác (chuỗi tương tự, phương trình bằng nhau).
4. **Lớn nhất theo từ điển** → sắp xếp giảm dần trong mỗi nhóm.
5. **Đếm số chuỗi khác nhau đạt được** → tích các hệ số đa thức (multinomial) của từng nhóm.

In [23]:
from math import factorial
from collections import Counter


def count_distinct_reachable(s, pairs):
    """Biến thể 5: số chuỗi khác nhau = tích (|nhóm|! / tích (tần suất!))."""
    n = len(s)
    dsu = DSU(n)
    for a, b in pairs:
        dsu.union(a, b)
    groups = defaultdict(list)
    for i in range(n):
        groups[dsu.find(i)].append(s[i])
    total = 1
    for chars in groups.values():
        ways = factorial(len(chars))
        for v in Counter(chars).values():
            ways //= factorial(v)
        total *= ways
    return total


def brute_count(s, pairs):
    seen, q = {s}, deque([s])
    while q:
        cur = q.popleft()
        for a, b in pairs:
            l = list(cur); l[a], l[b] = l[b], l[a]; nx = ''.join(l)
            if nx not in seen:
                seen.add(nx); q.append(nx)
    return len(seen)


rng = random.Random(15)
for _ in range(200):
    n = rng.randint(1, 6)
    s = ''.join(rng.choice('abc') for _ in range(n))
    pairs = [[rng.randrange(n), rng.randrange(n)] for _ in range(rng.randint(0, 4))]
    assert count_distinct_reachable(s, pairs) == brute_count(s, pairs)
print(count_distinct_reachable("dcab", [[0, 3], [1, 2]]), "| biến thể 5 OK")

4 | biến thể 5 OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS chuỗi | Liệt kê mọi trạng thái | `O(n!·…)` | `O(n!·n)` | Oracle `n ≤ 7` |
| 2. DFS (stack) | Thành phần + sort | `O(n + m + n log n)` | `O(n + m)` | Đã có sẵn đồ thị |
| 3. BFS | Như 2, dùng hàng đợi | `O(n + m + n log n)` | `O(n + m)` | Như 2 |
| 4. DSU + sort | Gộp nhóm theo cặp | `O((n+m)α + n log n)` | `O(n)` | Nộp bài, gọn nhất |
| 5. DSU + đếm | Counting sort 26 chữ | ~`O(n + m)` | `O(n)` | Cần nhanh tuyệt đối |

## 17. Bài tập vận dụng

**Bài 1 (đoán output):** `smallestStringWithSwaps("leetcode", [[0,7],[7,3]])` = ?

**Bài 2 (tính tay):** với `s = "zxyabc"`, `pairs = [[0,5],[5,2],[1,4]]`, liệt kê các nhóm và kết quả.

**Bài 3 (điền TODO):** hoàn thiện `components(n, pairs)` trả về list các nhóm (mỗi nhóm là list chỉ số tăng dần) bằng DSU.

**Bài 4 (tự viết lại):** viết phiên bản trả về chuỗi **lớn nhất** theo từ điển.

**Bài 5 (mini challenge):** LeetCode 1722 — cho `source`, `target` (list số) và `allowedSwaps`, trả về khoảng cách Hamming nhỏ nhất.

In [24]:
def components(n, pairs):
    # TODO: DSU, trả về list nhóm, mỗi nhóm tăng dần
    pass


def largest_string_with_swaps(s, pairs):
    # TODO
    raise NotImplementedError


def minimum_hamming_distance(source, target, allowedSwaps):
    # TODO
    raise NotImplementedError

# assert sorted(components(4, [[0, 3], [1, 2]])) == [[0, 3], [1, 2]]
# assert minimum_hamming_distance([1, 2, 3, 4], [2, 1, 4, 5], [[0, 1], [2, 3]]) == 1

## 18. Gợi ý 3 mức

- **Mức 1:** Coi chỉ số là đỉnh, cặp là cạnh. Điều gì giữ nguyên dưới mọi phép hoán đổi?
- **Mức 2:** Trong một thành phần liên thông, ký tự sắp xếp tùy ý được.
- **Mức 3:** Bài 5: với mỗi nhóm, `Counter` các giá trị `source`, rồi duyệt `target` trong nhóm: nếu còn trong Counter thì giảm, ngược lại tăng đáp án.

## 19. Đáp án tham khảo

**Bài 1:** nhóm `{0, 3, 7}` có ký tự `l, t, e` → `e, l, t` → vị trí 0=`e`, 3=`l`, 7=`t` → `"eeelcodt"`.

**Bài 2:** nhóm `{0,2,5}` ký tự `z,y,c` → `c,y,z`; nhóm `{1,4}` ký tự `x,b` → `b,x`; vị trí 3 giữ `a` → `"cbyaxz"`.

In [25]:
assert Solution().smallestStringWithSwaps("leetcode", [[0, 7], [7, 3]]) == "eeelcodt"
assert Solution().smallestStringWithSwaps("zxyabc", [[0, 5], [5, 2], [1, 4]]) == "cbyaxz"


def components(n, pairs):
    dsu = DSU(n)
    for a, b in pairs:
        dsu.union(a, b)
    groups = defaultdict(list)
    for i in range(n):
        groups[dsu.find(i)].append(i)
    return list(groups.values())


def largest_string_with_swaps(s, pairs):
    res = list(s)
    for idx in components(len(s), pairs):
        for i, ch in zip(idx, sorted((s[i] for i in idx), reverse=True)):
            res[i] = ch
    return ''.join(res)


def minimum_hamming_distance(source, target, allowedSwaps):
    ans = 0
    for idx in components(len(source), allowedSwaps):
        cnt = Counter(source[i] for i in idx)
        for i in idx:
            if cnt[target[i]] > 0:
                cnt[target[i]] -= 1
            else:
                ans += 1
    return ans


assert sorted(components(4, [[0, 3], [1, 2]])) == [[0, 3], [1, 2]]
assert minimum_hamming_distance([1, 2, 3, 4], [2, 1, 4, 5], [[0, 1], [2, 3]]) == 1
assert minimum_hamming_distance([5, 1, 2, 4, 3], [1, 5, 4, 2, 3], [[0, 4], [4, 2], [1, 3], [1, 4]]) == 0
rng = random.Random(19)
for _ in range(300):
    n = rng.randint(1, 6)
    s = ''.join(rng.choice('abcd') for _ in range(n))
    pairs = [[rng.randrange(n), rng.randrange(n)] for _ in range(rng.randint(0, 5))]
    seen, q = {s}, deque([s])
    while q:
        cur = q.popleft()
        for a, b in pairs:
            l = list(cur); l[a], l[b] = l[b], l[a]; nx = ''.join(l)
            if nx not in seen:
                seen.add(nx); q.append(nx)
    assert largest_string_with_swaps(s, pairs) == max(seen)
print("Đáp án bài tập: OK")

Đáp án bài tập: OK


## 20. Template nhớ nhanh + Checklist trước khi submit

```python
parent = list(range(n))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]   # nén nửa đường (path halving)
        x = parent[x]
    return x
for a, b in pairs: parent[find(a)] = find(b)
groups = defaultdict(list)
for i in range(n): groups[find(i)].append(i)      # i tăng dần
res = [''] * n
for idx in groups.values():
    for i, ch in zip(idx, sorted(s[i] for i in idx)): res[i] = ch
return ''.join(res)
```

**Checklist:**
- [ ] `pairs` rỗng → trả nguyên `s`.
- [ ] Cặp trùng / cặp `[i, i]` không gây lỗi.
- [ ] Chỉ số trong nhóm được **sắp tăng dần** trước khi ghép.
- [ ] Không dùng DFS đệ quy với `n = 10^5`.
- [ ] Kết quả ghép bằng `''.join`.